# 05 — Sensitivity study: every row of Table 3

**What this notebook does:** produces every number in the paper's sensitivity table. Each row of that table is a *pair* of TEMPUS runs differing along exactly
one axis, so the drift between them **is** that term's contribution — no external reference is
involved anywhere in this notebook.

**Why it is organised as a registry.** Several runs serve more than one row, and the DE440 lunar
runs already exist from notebook 04. Re-integrating them would waste minutes and risk two
notebooks disagreeing. So §1 declares every run once, points each at its CSV (existing or new),
and integrates only what is missing. Everything downstream reads those CSVs by path.

**The three blocks**

| block | question | axis varied |
|---|---|---|
| (a) | how do the model terms rank? | PN terms / small bodies / Sun $J_2$, Earth on INPOP19a |
| (b) | how many small bodies are needed, and what do the distant ones add? | perturber count, per body |
| (c) | how much does the ephemeris matter? | ephemeris, each with its own small-body set |

**Required data:** the INPOP19a/21a TDB `.dat` files, `de430.bsp`, `de440.bsp`, `epm2021.bsp` with
their `.tpc` and header files, and the small-body kernels `sb441-n373s.bsp` and `ast343de430.bsp`.

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

from tempus import TimeScalar
from tempus.output import read_csv
from nb_utils import run_tempus_integration

EPH  = Path("../data/ephem").resolve()
KDIR = Path("../data/kernels").resolve()
OUT  = Path("../output").resolve()

# TDB flavour of every release, so all runs share one timescale with the DE/EPM SPKs
eph_19a = EPH / "inpop19a_TDB_m100_p100_tt.dat"
eph_21a = EPH / "inpop21a_TDB_m100_p100_tt.dat"
eph_440 = [KDIR / "de440.bsp", KDIR / "gm_de440.tpc"]
eph_430 = [KDIR / "de430.bsp", KDIR / "gm_de431.tpc"]
eph_epm = [KDIR / "epm2021.bsp", KDIR / "epm2021.tpc"]

sb441   = KDIR / "sb441-n373s.bsp"          # 343 MBAs + 30 KBOs
ast430  = KDIR / "ast343de430.bsp"          # 343 MBAs for DE430
gm_19a  = EPH / "header_mba.19a"
gm_21a  = EPH / "header_mba.21a"
gm_440  = KDIR / "header.440"
gm_kbo  = KDIR / "header_kbo.440"
gm_430  = KDIR / "header.430_572"
gm_epm  = KDIR / "header.21h"

# The five asteroids that enter INPOP's Einstein-Infeld-Hoffmann equations of motion:
# Ceres, Pallas, Vesta, Iris, Bamberga.
BIG5 = "1,2,4,7,324"

# The trans-Neptunian objects INPOP19a and DE440 carry, named explicitly. 
# A simple numeric cut cannot separate them.
DISTANT = {
    50000:  "Quaoar",    55637:  "2002 UX25", 90482:  "Orcus",
    120347: "Salacia",   136108: "Haumea",    136199: "Eris",
    136472: "Makemake",  174567: "Varda",     208996: "2003 AZ84",
    225088: "Gonggong",
}


def resolved_asteroid_ids(csv_path):
    """Split the asteroids a run actually used into main-belt and distant.

    The ids come from the run's own `.run.json`, not from the GM header:
    `header_mba.19a` is a *supplement* to the ephemeris constants and omits
    MA1, MA2 and MA4 -- Ceres, Pallas and Vesta, the three most massive bodies in
    the belt, whose GMs TEMPUS reads from the ephemeris itself. Building the
    main-belt list from the header alone therefore silently drops exactly the
    objects that dominate the belt, and makes the main-belt-only run come out
    *weaker* than the five-asteroid one.
    """
    meta = json.loads(Path(str(csv_path).replace(".csv", ".run.json")).read_text())
    ids = meta["asteroid_indices"]
    return [i for i in ids if i not in DISTANT], [i for i in ids if i in DISTANT]


NS_CY = 3.168808781e-19          # 1 ns/cy in s/s

# One grid for every run: +-100 yr around J2000, in TDB
START = TimeScalar(2451545 - 36525, np.float64(0.0))
END   = TimeScalar(2451545 + 36525, np.float64(0.0))

## 1 · The run registry

Every run the table needs, declared once. `csv` points at an existing file where an earlier
notebook already produced it — those entries are reused, never recomputed: the DE440 lunar runs
(planets, +343 MBA, +373 bodies) come from notebook 04.

`model` holds the keyword arguments passed to `tempus integrate`; an empty dict is the
planets-only baseline (`--ext auto`).

In [2]:
def R(target, eph, csv, **model):
    return dict(target=target, eph=eph, csv=OUT / csv, model=model)

MBA  = dict(asteroid_file=sb441, asteroid_gm_header=gm_440)
RUNS = {
    # ---- Earth, INPOP19a --------------------------------------------------
    "E19_base":   R("EARTH_CENTER", eph_19a, "nb05_earth_200yr_19a.csv"),
    "E19_pn1":    R("EARTH_CENTER", eph_19a, "nb05_earth_200yr_19a_pn1.csv", pn1=True),
    "E19_353":    R("EARTH_CENTER", eph_19a, "nb05_earth_200yr_19a_353.csv",
                    asteroid_file=sb441, asteroid_gm_header=gm_19a),
    "E19_5mba":   R("EARTH_CENTER", eph_19a, "nb05_earth_200yr_19a_5mba.csv",
                    asteroid_file=sb441, asteroid_gm_header=gm_19a, asteroid_ids=BIG5),
    "E19_343":    R("EARTH_CENTER", eph_19a, "nb05_earth_200yr_19a_mbaonly.csv",
                    asteroid_file=sb441, asteroid_gm_header=gm_19a),
    "E19_j2":     R("EARTH_CENTER", eph_19a, "nb05_earth_200yr_19a_sunj2.csv", sun_j2=True),

    # ---- Mars, INPOP19a ---------------------------------------------------
    "M19_base":   R("MARS_BARYCENTER", eph_19a, "nb05_mars_200yr_19a_base.csv"),
    "M19_5mba":   R("MARS_BARYCENTER", eph_19a, "nb05_mars_200yr_19a_5mba.csv",
                    asteroid_file=sb441, asteroid_gm_header=gm_19a, asteroid_ids=BIG5),
    "M19_343":    R("MARS_BARYCENTER", eph_19a, "nb05_mars_200yr_19a_mbaonly.csv",
                    asteroid_file=sb441, asteroid_gm_header=gm_19a),
    # INPOP19a with its own small-body set, so also its row in block (c)
    "M19_353":    R("MARS_BARYCENTER", eph_19a, "nb05_mars_200yr_19a_own.csv",
                    asteroid_file=sb441, asteroid_gm_header=gm_19a),

    # ---- Moon, DE440 (reused from notebook 04) ----------------------------
    "L440_base":  R("MOON", eph_440, "nb04_moon_200yr_DE440_base.csv"),
    "L440_343":   R("MOON", eph_440, "nb04_moon_200yr_DE440_mba.csv", **MBA),
    "L440_373":   R("MOON", eph_440, "nb04_moon_200yr_DE440_all.csv",
                    asteroid_file=sb441, asteroid_gm_header=[gm_440, gm_kbo]),
    "L440_5mba":  R("MOON", eph_440, "nb05_moon_200yr_DE440_5mba.csv",
                    asteroid_file=sb441, asteroid_gm_header=gm_440, asteroid_ids=BIG5),

    # ---- Mars, ephemeris axis ---------------------------------------------
    # EPM2021 takes DE430's asteroid trajectories with its own GM values, since it
    # ships no trajectory file of its own.
    "M21_own":    R("MARS_BARYCENTER", eph_21a, "nb05_mars_200yr_21a_own.csv",
                    asteroid_file=sb441, asteroid_gm_header=gm_21a),
    "M430_own":   R("MARS_BARYCENTER", eph_430, "nb05_mars_200yr_DE430_own.csv",
                    asteroid_file=ast430, asteroid_gm_header=gm_430),
    "Mepm_own":   R("MARS_BARYCENTER", eph_epm, "nb05_mars_200yr_EPM2021_own.csv",
                    asteroid_file=ast430, asteroid_gm_header=gm_epm),
}

have = sum(r["csv"].is_file() for r in RUNS.values())
print(f"{have} of {len(RUNS)} runs already on disk; {len(RUNS) - have} to integrate.")
for k, r in RUNS.items():
    print(f"  {'have' if r['csv'].is_file() else 'MISSING':8s} {k:11s} {r['csv'].name}")

3 of 17 runs already on disk; 14 to integrate.
  MISSING  E19_base    nb05_earth_200yr_19a.csv
  MISSING  E19_pn1     nb05_earth_200yr_19a_pn1.csv
  MISSING  E19_353     nb05_earth_200yr_19a_353.csv
  MISSING  E19_5mba    nb05_earth_200yr_19a_5mba.csv
  MISSING  E19_343     nb05_earth_200yr_19a_mbaonly.csv
  MISSING  E19_j2      nb05_earth_200yr_19a_sunj2.csv
  MISSING  M19_base    nb05_mars_200yr_19a_base.csv
  MISSING  M19_5mba    nb05_mars_200yr_19a_5mba.csv
  MISSING  M19_343     nb05_mars_200yr_19a_mbaonly.csv
  MISSING  M19_353     nb05_mars_200yr_19a_own.csv
  have     L440_base   nb04_moon_200yr_DE440_base.csv
  have     L440_343    nb04_moon_200yr_DE440_mba.csv
  have     L440_373    nb04_moon_200yr_DE440_all.csv
  MISSING  L440_5mba   nb05_moon_200yr_DE440_5mba.csv
  MISSING  M21_own     nb05_mars_200yr_21a_own.csv
  MISSING  M430_own    nb05_mars_200yr_DE430_own.csv
  MISSING  Mepm_own    nb05_mars_200yr_EPM2021_own.csv


### Integrating what is missing

**This is the expensive cell.** It walks the registry and integrates only the entries whose CSV
is absent, so a repeat session prints a list of `have` lines and finishes immediately. Nothing
below depends on this cell's runtime state — the analysis reads the CSVs.

Every run uses the TDB flavour of its ephemeris on one grid (±100 yr around J2000 in TDB, 0.75 d
steps), so any two runs can be differenced at equal epoch numbers. The main-belt-only runs take
their asteroid ids from the all-bodies run `E19_353`, which the loop integrates first.

In [3]:
for key, r in RUNS.items():
    if r["csv"].is_file():
        print(f"  have {r['csv'].name}")
        continue
    if key in ("E19_343", "M19_343"):
        # main belt = what the all-bodies run resolved, minus the distant objects
        r["model"]["asteroid_ids"] = ",".join(str(i) for i in resolved_asteroid_ids(RUNS["E19_353"]["csv"])[0])
    print(f"\n=== integrating {key} -> {r['csv'].name} ===", flush=True)
    run_tempus_integration(
        r["eph"], START, END, jd_step=0.75, out=r["csv"], zero=True,
        target=r["target"], center="SSB", integrator="am12",
        **{"ext": "auto", **r["model"]},
    )

MBA_IDS, TNO_IDS = resolved_asteroid_ids(RUNS["E19_353"]["csv"])
print(f"\nINPOP19a resolved set: {len(MBA_IDS)} main-belt + {len(TNO_IDS)} distant")
print(f"  Ceres/Pallas/Vesta present: {[i for i in (1, 2, 4) if i in MBA_IDS]}")
print(f"  highest main-belt id: {max(MBA_IDS)}")
print(f"  distant: {', '.join(f'{i} {DISTANT[i]}' for i in TNO_IDS)}")


=== integrating E19_base -> nb05_earth_200yr_19a.csv ===
Ephemeris version: INPOP19A
CALCEPH timescale: 1 (TDB)
  zeroed: TCB_minus_TCX = 0 at JD 2443144 + 0.5003724992418982 (TDB) (shift -35.983421997695707 s)
Approx. mean drift (TCB-TCX): 1.279434 ms/day
Wrote /home/yan/PythonWorkdir/CNRS_Release/TEMPUS/output/nb05_earth_200yr_19a.csv

=== integrating E19_pn1 -> nb05_earth_200yr_19a_pn1.csv ===
Ephemeris version: INPOP19A
CALCEPH timescale: 1 (TDB)
  zeroed: TCB_minus_TCX = 0 at JD 2443144 + 0.5003724992418982 (TDB) (shift -35.983421731281574 s)
Approx. mean drift (TCB-TCX): 1.279434 ms/day
Wrote /home/yan/PythonWorkdir/CNRS_Release/TEMPUS/output/nb05_earth_200yr_19a_pn1.csv

=== integrating E19_353 -> nb05_earth_200yr_19a_353.csv ===
Ephemeris version: INPOP19A
CALCEPH timescale: 1 (TDB)
Loaded GM header 'header_mba.19a': 350 new (header values take priority).
Skipped 20 asteroids without GM constants.
Using 353 auto-discovered asteroids (positions + GM available).
  zeroed: TCB_mi

### Verifying what each run actually used

A requested asteroid id is silently dropped when the trajectory file has no segment for it or
no GM can be resolved, so the number of bodies a run *used* need not equal the number asked for.
The cell below reads that count back from each run's own `.run.json` and checks it against the
registry. This is the check that would have caught the main-belt list being built from the GM
header, which quietly omitted Ceres, Pallas and Vesta.

In [4]:
EXPECTED = {          # runs whose body count is known in advance
    "E19_5mba": 5, "M19_5mba": 5, "L440_5mba": 5,
    "E19_343": len(MBA_IDS), "M19_343": len(MBA_IDS),
    "E19_353": len(MBA_IDS) + len(TNO_IDS),
    "Mepm_own": 244, "M430_own": 343, "M21_own": 353,
    "L440_343": 343, "L440_373": 373, "M19_353": 353,
}

print(f"{'run':12s} {'used':>6s} {'expected':>9s}  status")
print("-" * 44)
bad = []
for key, r in RUNS.items():
    meta_path = Path(str(r["csv"]).replace(".csv", ".run.json"))
    if not meta_path.is_file():
        continue
    used = len(json.loads(meta_path.read_text()).get("asteroid_indices") or [])
    exp = EXPECTED.get(key)
    if exp is None:
        print(f"{key:12s} {used:6d} {'-':>9s}  - ")
        continue
    ok = used == exp
    bad += [] if ok else [key]
    print(f"{key:12s} {used:6d} {exp:9d}  {'ok' if ok else 'MISMATCH'}")

assert not bad, f"asteroid count mismatch in: {bad}"
print("\nall expected counts match")

run            used  expected  status
--------------------------------------------
E19_base          0         -  - 
E19_pn1           0         -  - 
E19_353         353       353  ok
E19_5mba          5         5  ok
E19_343         343       343  ok
E19_j2            0         -  - 
M19_base          0         -  - 
M19_5mba          5         5  ok
M19_343         343       343  ok
M19_353         353       353  ok
L440_base         0         -  - 
L440_343        343       343  ok
L440_373        373       373  ok
L440_5mba         5         5  ok
M21_own         353       353  ok
M430_own        343       343  ok
Mepm_own        244       244  ok

all expected counts match


## 2 · Differencing two runs

`compare(x, y)` is the whole method: align both series at their first sample (TCX−TCB is defined
only up to an origin), fit and remove a straight line, and report the slope as the *drift* and
the peak of what remains as the *max residual*. Those are the table's two columns.

The drift is quoted as a magnitude, matching the table.

In [5]:
rows = []


def series(key):
    """Epochs (TDB seconds past J2000) and TCX-TCB.

    All runs share one TDB grid, so any two can be differenced at equal epoch
    numbers. A TCB-scaled run differenced against a TDB-scaled one would compare
    different physical instants and inject a spurious drift of L_C x L_B = 2.3e-16
    s/s -- two orders above anything this table measures. The grid assertion in
    compare() is what enforces that this never happens.
    """
    c = read_csv(RUNS[key]["csv"])[0]
    return (np.asarray(c["SecondsPastJ2000"], float),
            -np.asarray(c["TCB_minus_TCX"], float))


def compare(kx, ky, label, block):
    """Drift and detrended peak between two runs; appended to the table."""
    etx, x = series(kx)
    ety, y = series(ky)
    n = min(len(x), len(y))
    et, x, y = etx[:n], x[:n], y[:n]
    if not np.allclose(etx[:n], ety[:n], atol=1e-3):
        raise ValueError(f"{kx} and {ky} are not on a common grid -- check they "
                         f"share a timescale")

    d = y - x # if we didn't use default IAU --zero at 1977 epoch: d = (y - y[0]) - (x - x[0])
    slope, icpt, *_ = stats.linregress(et, d)
    peak = np.max(np.abs(d - (slope * et + icpt)))
    rows.append((block, label, abs(slope), peak))
    print(f"[{block}] {label:52s} {abs(slope):.3e} s/s "
          f"({abs(slope) / NS_CY:8.2f} ns/cy)  peak {peak * 1e12:9.2f} ps")
    return abs(slope), peak

## 3 · Block (a) — the hierarchy of model terms

Earth on INPOP19a, three terms switched on in turn. This block exists to set the scale: it is
what tells the reader whether the small-body numbers in block (b) are large or small.

The solar oblateness (`--sun-j2`) adds the Sun's quadrupole to the potential $U$:

$$\Delta U_{J_2,\odot} = -\frac{GM_\odot\, J_{2,\odot}\, R_\odot^2}{r^3}\, P_2(\sin\phi), \qquad P_2(x) = \tfrac{1}{2}(3x^2 - 1),$$

with $r$ the Sun–target distance and $\phi$ the target's heliographic latitude. $J_{2,\odot}$ and
$R_\odot$ are read from the ephemeris constants (`J2SUN`, `RSUN`/`ASUN`).

In [6]:
compare("E19_pn1",  "E19_base", "PN1 -> PN2 (beta/c^4)",        "a");
compare("E19_base", "E19_353",  "planets -> +353 small bodies", "a");
compare("E19_base", "E19_j2",   "planets -> +Sun J2",           "a");

[a] PN1 -> PN2 (beta/c^4)                                1.096e-16 s/s (  345.99 ns/cy)  peak     33.84 ps
[a] planets -> +353 small bodies                         6.193e-18 s/s (   19.54 ns/cy)  peak    160.36 ps
[a] planets -> +Sun J2                                   2.081e-20 s/s (    0.07 ns/cy)  peak      0.63 ps


## 4 · Block (b) — how many small bodies, and what the distant ones add

Each body gets the same three-step ladder. The first two rows are cumulative from a planets-only
run and show how quickly the belt saturates; the third is the *increment* from adding the
distant population on top of the complete main belt, so it can be read directly against the two
rows above it rather than being buried in a cumulative total.

Getting that third row requires running the main belt *without* the distant objects. DE440 ships
the two sets as separate GM headers (`header.440`, `header_kbo.440`) so the split is free there.
INPOP19a ships one header carrying both, so the main-belt ids are passed explicitly via
`--asteroid-ids`. Those ids are taken from the all-bodies run's `.run.json` rather than from the
GM header: the header is a supplement that omits Ceres, Pallas and Vesta, whose GMs come from
the ephemeris constants, so building the list from it alone would drop the three bodies that
dominate the belt.

In [7]:
compare("E19_base", "E19_5mba", "Earth  planets -> +5 MBAs",        "b");
compare("E19_base", "E19_343",  "Earth  planets -> +343 MBAs",      "b");
compare("E19_343",  "E19_353",  "Earth  +343 MBAs -> +10 distant",   "b");

compare("M19_base", "M19_5mba", "Mars   planets -> +5 MBAs",        "b");
compare("M19_base", "M19_343",  "Mars   planets -> +343 MBAs",      "b");
compare("M19_343",  "M19_353",  "Mars   +343 MBAs -> +10 distant",   "b");

compare("L440_base", "L440_5mba", "Moon   planets -> +5 MBAs",     "b");
compare("L440_base", "L440_343",  "Moon   planets -> +343 MBAs",   "b");
compare("L440_343",  "L440_373",  "Moon   +343 MBAs -> +30 KBOs",  "b");

[b] Earth  planets -> +5 MBAs                            2.759e-18 s/s (    8.71 ns/cy)  peak     12.84 ps
[b] Earth  planets -> +343 MBAs                          4.342e-18 s/s (   13.70 ns/cy)  peak     12.67 ps
[b] Earth  +343 MBAs -> +10 distant                      1.851e-18 s/s (    5.84 ns/cy)  peak    161.49 ps
[b] Mars   planets -> +5 MBAs                            2.919e-18 s/s (    9.21 ns/cy)  peak     70.54 ps
[b] Mars   planets -> +343 MBAs                          4.578e-18 s/s (   14.45 ns/cy)  peak     72.32 ps
[b] Mars   +343 MBAs -> +10 distant                      1.852e-18 s/s (    5.84 ns/cy)  peak    157.04 ps
[b] Moon   planets -> +5 MBAs                            2.739e-18 s/s (    8.64 ns/cy)  peak     12.34 ps
[b] Moon   planets -> +343 MBAs                          4.711e-18 s/s (   14.87 ns/cy)  peak     12.77 ps
[b] Moon   +343 MBAs -> +30 KBOs                         2.645e-18 s/s (    8.35 ns/cy)  peak    187.45 ps


## 5 · Block (c) — changing the ephemeris, for Mars

Mars is the interesting target for this question: it sits closest to the main belt, so it is the
body whose time scale is most exposed to how an ephemeris models the small bodies, and its
sensitivity bounds what the choice of input costs for a Mars mission.

Each ephemeris is taken as a user would receive it, with its own native small-body set; for
INPOP19a that is the `M19_353` run of block (b). EPM2021 ships no asteroid trajectory file, so it
takes DE430's with its own GM values. A row therefore measures the *total* change on switching
input — planetary orbits and small-body model together — which is the quantity that bounds
ephemeris-induced uncertainty in practice.

In [8]:
compare("M19_353",  "M21_own",  "Mars  INPOP19a -> INPOP21a",  "c");
compare("M21_own",  "M430_own", "Mars  INPOP21a -> DE430",     "c");
compare("M21_own",  "Mepm_own", "Mars  INPOP21a -> EPM2021",   "c");
compare("M430_own", "Mepm_own", "Mars  DE430 -> EPM2021",      "c");

[c] Mars  INPOP19a -> INPOP21a                           3.638e-19 s/s (    1.15 ns/cy)  peak    608.16 ps
[c] Mars  INPOP21a -> DE430                              2.099e-18 s/s (    6.62 ns/cy)  peak   1420.97 ps
[c] Mars  INPOP21a -> EPM2021                            2.436e-18 s/s (    7.69 ns/cy)  peak    535.71 ps
[c] Mars  DE430 -> EPM2021                               3.370e-19 s/s (    1.06 ns/cy)  peak   1675.66 ps


## 6 · Summary — the table as printed in the paper

Grouped by block, in the order the table lists them. These are the values that populate
`tab:ephemeris`; anything still marked TODO there should be filled from here.

In [9]:
print(f"{'blk':4s} {'comparison':32s} {'drift [1e-18 s/s]':>9s} {'ns/cy':>10s} {'peak [ps]':>11s}")
print("-" * 95)
for block in ("a", "b", "c"):
    for b, label, drift, peak in rows:
        if b != block:
            continue
        print(f"{b:4s} {label:32s} {drift*1e18:17.2f} {drift / NS_CY:10.2f} {peak * 1e12:11.0f}")
    print()

blk  comparison                       drift [1e-18 s/s]      ns/cy   peak [ps]
-----------------------------------------------------------------------------------------------
a    PN1 -> PN2 (beta/c^4)                       109.64     345.99          34
a    planets -> +353 small bodies                  6.19      19.54         160
a    planets -> +Sun J2                            0.02       0.07           1

b    Earth  planets -> +5 MBAs                     2.76       8.71          13
b    Earth  planets -> +343 MBAs                   4.34      13.70          13
b    Earth  +343 MBAs -> +10 distant               1.85       5.84         161
b    Mars   planets -> +5 MBAs                     2.92       9.21          71
b    Mars   planets -> +343 MBAs                   4.58      14.45          72
b    Mars   +343 MBAs -> +10 distant               1.85       5.84         157
b    Moon   planets -> +5 MBAs                     2.74       8.64          12
b    Moon   planets -> +343 MBAs  